RAG Config Lab — Chunking × Embedding × Retrieval, Proven by Numbers

Build a RAG knowledge layer over 5 materials-science papers, then empirically find and prove the best chunking + embedding + retrieval config against a frozen golden question set.

You already have the basic pipeline. This is the experiment harness on top of it. The thesis:


Your chunking strategy is your product's answer quality. This tutorial makes that claim measurable — every config is scored on the same questions, so "which is best" stops being an opinion and becomes a number.


Everything you'd swap — paths, models, chunk sizes, the golden set — lives in one CONFIG block. Change it, re-run, get a new ranking.

In [4]:
__import__("pysqlite3")
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

import os, re, time, uuid
from collections import Counter
import pandas as pd

from langchain_community.document_loaders import PyPDFDirectoryLoader   # community: sunset, but still the only home for this loader
from langchain_text_splitters import (RecursiveCharacterTextSplitter,
                                       CharacterTextSplitter)
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever              # community in 1.x
from langchain_classic.retrievers import EnsembleRetriever            # MOVED to langchain_classic in 1.x

import chromadb

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before the embedding steps."

1. CONFIG — the only block you edit

Everything downstream reads from here.

In [5]:
PDF_DIR = "papers"

# embedding models to compare: name -> (model_id, dims, ctx_limit, usd_per_1M_tokens)
EMBED_MODELS = {
    "3-small": ("text-embedding-3-small", 1536, 8191, 0.02),
    "3-large": ("text-embedding-3-large", 3072, 8191, 0.13),
}

CHUNK_SIZES   = [300, 800]     # two sizes
CHUNK_OVERLAP = 100
K             = 3              # top-k passed to the LLM
HYBRID_WEIGHTS = [0.4, 0.6]    # [BM25, dense] for EnsembleRetriever

# GOLDEN SET: (question, expected_paper). expected_paper = PDF filename stem.
# Grounded in the actual content/contributions of each paper.
GOLDEN = [
    # --- a-lab.pdf : autonomous robotic synthesis lab ---
    ("How does the autonomous lab decide which synthesis recipes to try?",        "a-lab"),
    ("How many novel inorganic compounds did the autonomous laboratory make?",    "a-lab"),
    ("What role does active learning play in robotic materials synthesis?",       "a-lab"),

    # --- chemcrow.pdf : LLM augmented with chemistry tools ---
    ("How are chemistry tools integrated into a large language model?",           "chemcrow"),
    ("Which agent uses external tools to plan and execute chemical syntheses?",   "chemcrow"),

    # --- mace-mp-0.pdf : MACE foundation model ---
    ("What is the MACE foundation model for atomistic materials chemistry?",      "mace-mp-0"),
    ("Which model uses higher-order equivariant message passing for force fields?","mace-mp-0"),

    # --- omat-2024.pdf : OMat24 dataset ---
    ("How many DFT calculations does the OMat24 dataset contain?",                "omat-2024"),
    ("What is systematic softening and how does dataset diversity reduce it?",    "omat-2024"),
    ("Which dataset samples non-equilibrium structures via rattling and AIMD?",   "omat-2024"),

    # --- orb3.pdf : Orb-v3 potentials ---
    ("Can non-conservative, non-equivariant potentials predict phonons well?",    "orb3"),
    ("Which model gives a 10x latency and 8x memory reduction over other MLIPs?", "orb3"),
    ("What is equigrad regularization?",                                          "orb3"),

    # --- universalmodelofatoms.pdf : UMA ---
    ("What is the mixture-of-linear-experts design used in UMA?",                 "universalmodelofatoms"),
    ("Which model was trained on around half a billion atomic structures?",       "universalmodelofatoms"),
]

2. Ingestion — parse PDFs, extract sections

Scientific papers have structure: abstract / methods / results / discussion answer different kinds of questions. A methods question ("how are bond angles encoded?") should retrieve a methods chunk, not an abstract. So we detect section headers and tag each span — this tagging is what makes section-aware chunking and metadata filtering possible later.

In [3]:
SECTIONS = ["abstract", "introduction", "methods", "results", "discussion", "conclusion"]

def split_sections(page_text):
    """Yield (section_type, text) by scanning for known headers. A real parser would
    use font size and numbering; this keyword scan is the honest minimum — swap it out."""
    cur, buf, out = "header", [], []
    for ln in page_text.split("\n"):
        low = ln.strip().lower()
        if low in SECTIONS:
            if buf: out.append((cur, " ".join(buf))); buf = []
            cur = low
        elif ln.strip():
            buf.append(ln.strip())
    if buf: out.append((cur, " ".join(buf)))
    return out

raw_pages = PyPDFDirectoryLoader(PDF_DIR).load()
def stem(d): return d.metadata["source"].split("/")[-1].replace(".pdf", "")

print(f"Loaded {len(raw_pages)} pages from {len({stem(d) for d in raw_pages})} papers")

Loaded 292 pages from 6 papers


In [6]:
print(raw_pages[0])

page_content='Augmenting large language models with chemistry tools
Andres M. Bran12∗ Sam Cox3∗ Oliver Schilter24
Carlo Baldassari4 Andrew D. White3 Philippe Schwaller12
1 Laboratory of Artificial Chemical Intelligence (LIAC), ISIC, EPFL
2National Centre of Competence in Research (NCCR) Catalysis, EPFL
3 Department of Chemical Engineering, University of Rochester
4 Accelerated Discovery, IBM Research – Europe
∗Contributed equally.
andrew.white@rochester.edu
philippe.schwaller@epfl.ch
Abstract
Over the last decades, excellent computational chemistry tools have been developed.
Integrating them into a single platform with enhanced accessibility could help reaching
their full potential by overcoming steep learning curves. Recently, large-language models
(LLMs) have shown strong performance in tasks across domains, but struggle with
chemistry-related problems. Moreover, these models lack access to external knowledge
sources, limiting their usefulness in scientific applications. In this stud

3. Metadata — tag every chunk

Each chunk carries paper and section_type, plus fields you'd pull from your own catalog. Metadata enables filtered retrieval ("answer only from methods sections of 2021+ papers") — often a bigger quality lever than the embedding model.

In [7]:
# PAPER_META: catalog metadata keyed by PDF filename stem.
PAPER_META = {
    "a-lab": {
        "year": 2023, "authors": "Szymanski et al.",
        "venue": "Nature", "method_type": "autonomous synthesis",
    },
    "chemcrow": {
        "year": 2023, "authors": "Bran et al.",
        "venue": "Nature Machine Intelligence", "method_type": "LLM agent",
    },
    "mace-mp-0": {
        "year": 2024, "authors": "Batatia et al.",
        "venue": "arXiv", "method_type": "foundation potential",
    },
    "omat-2024": {
        "year": 2026, "authors": "Barroso-Luque et al.",
        "venue": "arXiv", "method_type": "dataset",
    },
    "orb3": {
        "year": 2025, "authors": "Rhodes, Vandenhaute et al.",
        "venue": "arXiv", "method_type": "foundation potential",
    },
    "universalmodelofatoms": {
        "year": 2025, "authors": "Wood et al.",
        "venue": "arXiv", "method_type": "foundation potential",
    },
}

def enrich(paper, section_type):
    m = {"paper": paper, "section_type": section_type}
    m.update(PAPER_META.get(paper, {}))
    return m

4. Chunking strategies (≥3) × sizes (2)

Three strategies with genuinely different behavior:


fixed-size (CharacterTextSplitter): blind cut every N chars. The dumb baseline — severs ideas mid-sentence.
recursive (RecursiveCharacterTextSplitter): cut at the most natural boundary (para → line → sentence) that fits. The usual default.
section-aware: never cross a section boundary; sub-split only oversized sections. Each chunk is one coherent section-scoped idea, tagged with its section_type.


Every strategy carries the metadata from step 3, so retrieval quality is the only thing that varies.

In [8]:
def make_fixed(size, overlap):
    sp = CharacterTextSplitter(chunk_size=size, chunk_overlap=overlap, separator="")
    return [Document(page_content=c, metadata=enrich(stem(d), "mixed"))
            for d in raw_pages for c in sp.split_text(d.page_content)]

def make_recursive(size, overlap):
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    return [Document(page_content=c, metadata=enrich(stem(d), "mixed"))
            for d in raw_pages for c in sp.split_text(d.page_content)]

def make_section_aware(size, overlap):
    """One chunk per section; sub-split only sections longer than `size`."""
    sub = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    out = []
    for d in raw_pages:
        for stype, text in split_sections(d.page_content):
            for p in (sub.split_text(text) if len(text) > size else [text]):
                out.append(Document(page_content=p, metadata=enrich(stem(d), stype)))
    return out

CHUNKERS = {"fixed": make_fixed, "recursive": make_recursive, "section": make_section_aware}

for name, fn in CHUNKERS.items():
    for size in CHUNK_SIZES:
        print(f"{name:9s} size={size:<4d} -> {len(fn(size, CHUNK_OVERLAP)):3d} chunks")

fixed     size=300  -> 3991 chunks
fixed     size=800  -> 1253 chunks
recursive size=300  -> 3899 chunks
recursive size=800  -> 1237 chunks
section   size=300  -> 3987 chunks
section   size=800  -> 1255 chunks


5. Embedding models (≥2) — dimensions, context, cost

text-embedding-3-small (1536-dim, $0.02/1M) vs text-embedding-3-large (3072-dim, $0.13/1M — 6.5×). Large scores ~2 points higher on MTEB but doubles storage. The empirical question this lab answers: is that worth it for these papers? Embedding this whole corpus costs a fraction of a cent, so re-running the grid is free in practice.

In [9]:
_client = chromadb.Client()   # in-memory; nothing persisted between runs

def build_dense_vs(chunks, embed_model_id):
    emb = OpenAIEmbeddings(model=embed_model_id)
    return Chroma.from_documents(chunks, emb,
                                 collection_name="c_" + uuid.uuid4().hex[:10],
                                 client=_client)

def approx_cost(chunks, usd_per_1M):
    tokens = sum(len(c.page_content) for c in chunks) / 4      # ~4 chars/token
    return tokens / 1e6 * usd_per_1M

6. Retrieval modes — dense, BM25, hybrid

dense: vector similarity — strong on paraphrase, weak on exact tokens like 0.039 eV/atom.
BM25: lexical — strong on exact tokens/identifiers, blind to synonyms. No embedding cost.
hybrid: EnsembleRetriever merges both.

In [10]:
def build_bm25(chunks, k=K):
    r = BM25Retriever.from_documents(chunks); r.k = k
    return r

def build_hybrid(chunks, vs, k=K):
    dense = vs.as_retriever(search_kwargs={"k": k})
    return EnsembleRetriever(retrievers=[build_bm25(chunks, k), dense], weights=HYBRID_WEIGHTS)

Correctness note: LangChain's EnsembleRetriever uses weighted round-robin fusion, not strict Reciprocal Rank Fusion, even though its reputation (and many tutorials) call it RRF. Fine for this comparison — just don't mislabel it in your writeup. For true RRF, use LlamaIndex's QueryFusionRetriever(mode="reciprocal_rerank") or implement 1/(k+rank) yourself.

7. Validation — the core, not an afterthought

Metrics, all computed against the frozen GOLDEN set so every config is comparable:


precision@k — fraction of retrieved chunks from the right paper

recall@k — did any top-k chunk come from the right paper (1 relevant paper per question here)

MRR — 1/rank of the first correct hit; rewards putting the right chunk first

In [11]:
def is_relevant(doc, gold_paper):
    return doc.metadata.get("paper") == gold_paper

def score(retriever, k=K):
    P = R = M = 0.0
    for q, gold in GOLDEN:
        res = retriever.invoke(q)[:k]
        flags = [is_relevant(d, gold) for d in res]
        P += sum(flags) / k
        R += 1.0 if any(flags) else 0.0
        M += next((1/i for i, f in enumerate(flags, 1) if f), 0.0)
    n = len(GOLDEN)
    return {"P@k": P/n, "R@k": R/n, "MRR": M/n}

8. Run the full grid

3 chunkers × 2 sizes × 2 embedders for dense, plus BM25 (embedding-free) and hybrid on the section chunks. Every row scored on the same questions.

In [12]:
rows = []

for chunker_name, fn in CHUNKERS.items():
    for size in CHUNK_SIZES:
        chunks = fn(size, CHUNK_OVERLAP)
        for embed_name, (mid, dims, ctx, price) in EMBED_MODELS.items():
            vs = build_dense_vs(chunks, mid)
            t0 = time.time()
            m = score(vs.as_retriever(search_kwargs={"k": K}))
            rows.append({"chunking": chunker_name, "size": size, "embed": embed_name,
                         "dims": dims, "retrieval": "dense", **m,
                         "latency_s": round(time.time()-t0, 2),
                         "index_cost_$": round(approx_cost(chunks, price), 5)})



In [13]:
# BM25 + hybrid on section chunks (extend to the full grid if you want)
sec = make_section_aware(CHUNK_SIZES[0], CHUNK_OVERLAP)
rows.append({"chunking": "section", "size": CHUNK_SIZES[0], "embed": "-", "dims": 0,
             "retrieval": "bm25", **score(build_bm25(sec)), "latency_s": 0.0, "index_cost_$": 0.0})

vs_small = build_dense_vs(sec, EMBED_MODELS["3-small"][0])
rows.append({"chunking": "section", "size": CHUNK_SIZES[0], "embed": "3-small", "dims": 1536,
             "retrieval": "hybrid", **score(build_hybrid(sec, vs_small)),
             "latency_s": 0.0, "index_cost_$": round(approx_cost(sec, 0.02), 5)})

df = pd.DataFrame(rows)

9. Results table — ranked

In [14]:
df_ranked = df.sort_values(["MRR", "P@k", "R@k"], ascending=False).reset_index(drop=True)
df_ranked

,chunking,size,embed,dims,retrieval,P@k,R@k,MRR,latency_s,index_cost_$
0,fixed,800,3-small,1536,dense,0.911111,1.000000,0.966667,3.86,0.00447
1,recursive,800,3-small,1536,dense,0.866667,1.000000,0.966667,4.05,0.00417
2,section,800,3-small,1536,dense,0.888889,1.000000,0.933333,4.19,0.00444
3,section,800,3-large,3072,dense,0.888889,1.000000,0.933333,6.87,0.02887
4,fixed,300,3-large,3072,dense,0.844444,1.000000,0.933333,6.22,0.03794
5,section,300,3-large,3072,dense,0.888889,1.000000,0.922222,7.75,0.03728
6,recursive,300,3-small,1536,dense,0.844444,1.000000,0.922222,4.02,0.00470
7,fixed,800,3-large,3072,dense,0.888889,1.000000,0.900000,6.44,0.02906
8,recursive,300,3-large,3072,dense,0.866667,1.000000,0.900000,6.14,0.03056
9,recursive,800,3-large,3072,dense,0.866667,1.000000,0.900000,5.69,0.02714


10. Pick the winner — justified by the numbers

In [15]:
best = df_ranked.iloc[0]
print("BEST CONFIG for this corpus")
print(f"  chunking : {best['chunking']} @ size {best['size']}")
print(f"  embedding: {best['embed']} ({best['dims']}d)")
print(f"  retrieval: {best['retrieval']}")
print(f"  P@{K}={best['P@k']:.3f}  R@{K}={best['R@k']:.3f}  MRR={best['MRR']:.3f}")

# Does 3-large actually earn its 6.5x cost? Same chunking, small vs large.
for chk in df_ranked["chunking"].unique():
    sub = df_ranked[(df_ranked.chunking == chk) & (df_ranked.retrieval == "dense")]
    s, l = sub[sub.embed == "3-small"], sub[sub.embed == "3-large"]
    if len(s) and len(l):
        ds, dl = s.iloc[0], l.iloc[0]
        verdict = "worth it" if dl["MRR"] > ds["MRR"] + 0.05 else "NOT worth 6.5x"
        print(f"[{chk}] 3-small MRR={ds['MRR']:.3f} vs 3-large MRR={dl['MRR']:.3f} -> {verdict}")

BEST CONFIG for this corpus
  chunking : fixed @ size 800
  embedding: 3-small (1536d)
  retrieval: dense
  P@3=0.911  R@3=1.000  MRR=0.967
[fixed] 3-small MRR=0.967 vs 3-large MRR=0.933 -> NOT worth 6.5x
[recursive] 3-small MRR=0.967 vs 3-large MRR=0.900 -> NOT worth 6.5x
[section] 3-small MRR=0.933 vs 3-large MRR=0.933 -> NOT worth 6.5x


In [16]:
emb = OpenAIEmbeddings(model=EMBED_MODELS["3-small"][0])
sec = make_section_aware(300, CHUNK_OVERLAP)
vs = Chroma.from_documents(sec, emb, collection_name="filt_" + uuid.uuid4().hex[:6], client=_client)

q = "How are bond angles represented?"
print("UNFILTERED:")
for d in vs.similarity_search(q, k=3):
    print(f"  [{d.metadata['paper']}/{d.metadata['section_type']}] {d.page_content[:60]}")

print("\nFILTERED to section_type='methods':")
for d in vs.similarity_search(q, k=3, filter={"section_type": "methods"}):
    print(f"  [{d.metadata['paper']}/{d.metadata['section_type']}] {d.page_content[:60]}")

UNFILTERED:
  [mace-mp-0/header] [ps] 140 160 180OB H O [°] d3 a 0 20 40 60 80 100 120 time [
  [mace-mp-0/header] value of2.27Å (77) and the value of 2.23Å from the custom ML
  [mace-mp-0/header] snapshots from simulations with either potential. (d) Coordi

FILTERED to section_type='methods':
  [a-lab/methods] that were marked as ‘theoretical’ (that is, not represented 
  [a-lab/methods] carbonates, bicarbo- nates, hydroxides, sulfates, sulfites, 
  [a-lab/methods] enthalpies of formation and standard Gibbs free energies of 


12. Optional — cross-encoder reranker

Hybrid boosts recall (right chunks in the candidate set); a reranker fixes order (right chunk first), which is what MRR and answer quality reward. Retrieve wide (k=20), then re-score each (query, chunk) pair jointly.

In [17]:
# pip install flashrank
# from flashrank import Ranker, RerankRequest
# ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")
#
# def rerank(query, docs, top_n=K):
#     passages = [{"id": i, "text": d.page_content} for i, d in enumerate(docs)]
#     ranked = ranker.rerank(RerankRequest(query=query, passages=passages))
#     return [docs[r["id"]] for r in ranked[:top_n]]
#
# wide = vs.as_retriever(search_kwargs={"k": 20})
# q = "What MAE does SchNet reach on QM9?"
# print([d.metadata["paper"] for d in rerank(q, wide.invoke(q))])

Exercise


Run the grid as-is on your 5 PDFs. Which config wins on MRR? Is it what you'd have guessed?
Add 5 more golden pairs targeting methods questions specifically ("how does X compute Y?"). Does section-aware chunking pull ahead once the questions demand a specific section?
Tighten is_relevant to require the correct section_type, not just the correct paper. Watch precision@k become meaningful (and most configs get worse — that's the honesty of a stricter metric).
Set HYBRID_WEIGHTS = [0.7, 0.3] (BM25-heavy). For a corpus full of exact values and formulas, does leaning lexical beat the 0.4/0.6 default?
Stretch: uncomment the reranker, retrieve at k=20, rerank to k=3, and add it as a new row. Does it beat plain hybrid on MRR by enough to justify the extra model and latency?


What good looks like: you can name the single best config for these papers and point at the column of numbers that proves it — and when someone asks "why not 3-large?", you answer with a delta, not a hunch.